# gl-yolo — sweep complet (MVA → DOTA → VisDrone)

Entraîne les 8 architectures sur les 3 datasets (24 runs), 200 epochs, imgsz=1280.

**Avant de lancer** : Runtime → Change runtime type → GPU (A100 si disponible, sinon L4).

**Ce run va prendre plusieurs jours.** Le repo est cloné sur le disque local éphémère (rapide pour `git` -- cloné sur Drive, chaque `git pull`/`status` devient très lent car Drive est monté en FUSE). Seuls `runs/` et `logs/` sont liés vers Drive par lien symbolique pour survivre aux déconnexions de session : à chaque reconnexion, ré-exécutez les cellules dans l'ordre -- le repo est re-cloné en quelques secondes, les liens vers `runs/`/`logs/` sont recréés automatiquement, et `run_gpu.py` saute les modèles déjà terminés et reprend celui interrompu depuis son dernier checkpoint. Les datasets, eux, sont automatiquement téléchargés sur le disque local (pas Drive) par `train.py` lui-même -- pas de cellule à se souvenir de ré-exécuter pour ça.

## 1. Vérifier le GPU attribué

In [ ]:
!nvidia-smi
import torch
print("CUDA disponible:", torch.cuda.is_available())

## 2. Monter Google Drive (persistance entre sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Cloner (ou mettre à jour) le repo en local + lier runs/logs sur Drive

In [ ]:
import os

REPO_DIR = "/content/gl-yolo"                        # code en local : git reste rapide
PERSIST_DIR = "/content/drive/MyDrive/gl-yolo-runs"   # runs/ et logs/ persistent ici entre sessions

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/osiastossou/gl-yolo.git "{REPO_DIR}"
else:
    print("Repo déjà présent en local, mise à jour...")
    !git -C "{REPO_DIR}" pull

os.makedirs(f"{PERSIST_DIR}/runs", exist_ok=True)
os.makedirs(f"{PERSIST_DIR}/logs", exist_ok=True)
!ln -sfn "{PERSIST_DIR}/runs" "{REPO_DIR}/runs"
!ln -sfn "{PERSIST_DIR}/logs" "{REPO_DIR}/logs"

%cd {REPO_DIR}

## 4. Installer les dépendances

In [ ]:
!pip install -q -e .

## 5. (Optionnel) vérifier l'état d'un sweep précédent

Utile après une reconnexion, pour voir ce qui est déjà fini avant de relancer.

In [ ]:
!find runs -name best.pt 2>/dev/null
!find runs -name last.pt 2>/dev/null

## 6. Lancer le sweep complet — depuis le **Terminal**, pas une cellule

MVA (bird, single-class) en premier, puis DOTA, puis VisDrone — 8 architectures chacun.

**Important** : un process lancé en arrière-plan depuis une cellule du notebook (`subprocess.Popen`, `!cmd &`, ...) est classé par Colab dans le cgroup `jupyter-children`, qui a une limite RAM bien plus basse que la RAM totale affichée -- un entraînement YOLO la dépasse vite et se fait tuer (`Memory cgroup out of memory`, code -9), peu importe la taille du modèle. Un Terminal (Pro/Pro+, icône dans la barre latérale gauche ou Tools → Terminal) tourne comme un process indépendant, hors de ce cgroup restreint.

Dans le **Terminal** (pas ici) :
```bash
cd /content/gl-yolo
nohup python run_gpu.py -epochs 200 -imgsz 1280 > sweep_colab.log 2>&1 &
disown
```
Le notebook ne sert plus qu'à surveiller (section 7) -- il ne lance plus rien lui-même.

In [19]:
!python analyze_runs.py

2 run(s): Y11-MVA1280, Y12-MVA1280

        run model dataset  epochs_done  best_epoch  precision  recall  mAP50  mAP50-95  total_time_h  min_per_epoch
Y11-MVA1280   Y11     MVA          201         201     0.8841  0.7298 0.7796    0.3596       18.2236         5.4399
Y12-MVA1280   Y12     MVA           57          57     0.8708  0.6816 0.7321    0.3087        3.0841         3.2464

Résumé -> /Users/osias/Documents/PHD/ultralytics-osias/analysis/summary.csv

Graphiques:
  -> /Users/osias/Documents/PHD/ultralytics-osias/analysis/losses_MVA.png
  -> /Users/osias/Documents/PHD/ultralytics-osias/analysis/metrics_MVA.png
  -> /Users/osias/Documents/PHD/ultralytics-osias/analysis/time_MVA.png
  -> /Users/osias/Documents/PHD/ultralytics-osias/analysis/compare_mAP50-95_MVA.png


In [ ]:
%cd /Users/osias/Documents/PHD/ultralytics-osias/
!python analyze_runs.py --datasets MVA --models Y11,Y12


/Users/osias/Documents/PHD/ultralytics-osias
usage: analyze_runs.py [-h] [--datasets DATASETS] [--models MODELS]
                       [--outdir OUTDIR] [--watch WATCH]
analyze_runs.py: error: unrecognized arguments: Y11GL, Y11CCP


In [16]:
!python evaluate_best.py --datasets DT --models Y11,Y12,Y11SCC,Y11GL,Y11CCP

Device: mps

=== Y11-DT1280 (DT, Y11) — split=val ===
Ultralytics 8.4.33 🚀 Python-3.9.18 torch-2.7.1 MPS (Apple M3 Pro)
YOLO11 summary (fused): 101 layers, 2,585,272 parameters, 0 gradients

WARNING ⚠️ Dataset 'ultralytics/cfg/datasets/DOTAv1.5.yaml' images not found, missing path '/Users/osias/Documents/PHD/ultralytics-osias/datasets/DOTAv1.5/images/val'
  File "/Users/osias/Documents/PHD/ultralytics-osias/ultralytics/engine/validator.py", line 185, in __call__
    self.data = check_det_dataset(self.args.data)
  File "/Users/osias/Documents/PHD/ultralytics-osias/ultralytics/data/utils.py", line 481, in check_det_dataset
    safe_download(url=s, dir=DATASETS_DIR, delete=True)
  File "/Users/osias/Documents/PHD/ultralytics-osias/ultralytics/utils/downloads.py", line 353, in safe_download
    data = response.read(buffer_size)
  File "/opt/anaconda3/envs/yolov10/lib/python3.9/http/client.py", line 463, in read
    n = self.readinto(b)
  File "/opt/anaconda3/envs/yolov10/lib/python3.9/http

## 7. Suivre la progression

Ré-exécutez cette cellule à tout moment pour voir : quel job tourne, l'epoch en cours et le détail du batch courant, et les métriques déjà sauvegardées pour ce job. Elle ne fait que lire des fichiers écrits par le process du Terminal -- aucun risque de la relancer.

In [ ]:
import glob
import os

print("--- Sweep : dernier job démarré/terminé ---")
!tail -5 sweep_colab.log 2>/dev/null

log_files = glob.glob("logs/*.log")
if not log_files:
    print("\nPas encore de log -- le premier job n'a pas encore démarré.")
else:
    latest = max(log_files, key=os.path.getmtime)
    name = os.path.splitext(os.path.basename(latest))[0]
    print(f"\n--- Job en cours : {name} (epoch/batch courant) ---")
    # le log contient des retours chariot (\r) pour la barre de progression -> on les
    # convertit en vrais saut de lignes pour voir le dernier état proprement
    !tail -c 4000 "{latest}" | tr '\r' '\n' | tail -5

    csv_path = f"runs/{name}/results.csv"
    if os.path.exists(csv_path):
        import pandas as pd

        print(f"\n--- Epochs déjà sauvegardés ({csv_path}) ---")
        print(pd.read_csv(csv_path).tail(5).to_string(index=False))
    else:
        print("\nPas encore de checkpoint sauvegardé pour ce job (results.csv pas encore créé).")

## En cas de déconnexion

Reconnectez-vous, ré-exécutez les cellules **2 à 6** dans l'ordre (Drive → repo local + liens runs/logs → deps → sweep). Le repo se re-clone en quelques secondes (il est en local, pas sur Drive), et les liens vers `runs/`/`logs/` sur Drive sont recréés automatiquement par la cellule 3. `run_gpu.py` saute les modèles déjà terminés (`best.pt` présent) et reprend automatiquement celui qui était en cours (`last.pt` sans `best.pt`) depuis son dernier checkpoint. Le dossier dataset se re-télécharge en local à chaque nouvelle session (quelques minutes) -- normal, c'est voulu.